# Task 1, Steps 5-6: Q1 Uniform PTQ and Q2 AWQ-style PTQ (8 / 4 / 3 bits, group 128)

**Quantizer (fixed for Q1/Q2/Q3, see `src/quant.py`):** groups of 128 consecutive input columns per output row;
asymmetric min/max range, no clipping; FP16 scale s = (max-min)/(2^b-1); zero-point z = round(-min/s) clipped to
[0, 2^b-1] and stored with b bits; round-half-to-even; codes and zero-points packed as little-endian bitstreams.
No padding is needed (1536 and 8960 are multiples of 128). Numerical evaluation uses reconstructed FP16 weights
(fake quantization, dense FP16 kernels); packed bytes are computed separately and verified by real packing.

**Q2 (AWQ-style):** per-input-channel scale s = mean|x|^alpha (normalised), alpha chosen from a 20-point grid
{0, 0.05, ..., 0.95} to minimise the calibration output error of the scaled+quantized group, computed exactly
from the 128 x 512 calibration Gram matrix. Inverse scales are folded into input_layernorm (q/k/v),
post_attention_layernorm (gate/up) and up_proj rows (down). o_proj cannot be folded (GQA: v_proj is 256-d, o_proj
reads 1536-d), so its compensating FP16 channel scale is stored explicitly (1536 x 2 bytes per block, counted).
Reproduced from AWQ: activation-derived scales, grid search on calibration activations, folding. Not reproduced:
module-output objective, clipping search, the AWQ inference engine / packed kernels.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task1_compression")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task1", need_public=True, need_prefix_banks=False)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from shared.src.io import save_json, load_json
from shared.src.plotting import save_figure
from task1_compression.src.runner import setup_task1
from task1_compression.src import quant, awq
from task1_compression.src.model_utils import layers_of, proj_module

T = setup_task1(ctx)
model = T.model
BITS = (8, 4, 3)
O_MODE = "explicit"

## Packing check on real projections
Real bit-packing of a few matrices must give exactly the closed-form byte count used for the whole model.

In [ ]:
rows = []
for p in ("q_proj", "k_proj", "down_proj"):
    W = proj_module(layers_of(model)[0], p).weight.data
    for b in (2, 3, 4, 8):
        ser = quant.serialize_matrix(W, b)
        cf = quant.matrix_packed_bytes(*W.shape, b)
        assert ser == cf
        rows.append(dict(proj=p, bits=b, shape=tuple(W.shape), **ser, fp16_bytes=W.numel() * 2))
pd.DataFrame(rows)

## Q1: uniform group-wise PTQ

In [ ]:
for b in BITS:
    ua = quant.uniform_assignment(model, b)
    T.runner.run(f"q1_int{b}",
                 apply_fn=lambda m, ua=ua: (quant.quantize_model_(m, ua), {})[1],
                 storage_fn=lambda m, st, ua=ua: dict(packed=quant.model_packed_bytes(m, ua)),
                 meta=dict(method="uniform asymmetric group-wise PTQ", bits=b, group_size=128, spec=quant.SPEC.as_dict()))

## AWQ scale search (calibration activations only; all bit widths incl. 2 for Q3)

In [ ]:
scale_path = ctx.checkpoints / "awq_scales.pt"
log_path = ctx.results / "awq_search_log.json"
if scale_path.exists():
    search = torch.load(scale_path)
else:
    T.snap.restore(model)
    search = awq.search_awq_scales(model, T.calib, bits_list=(2, 3, 4, 8), o_mode=O_MODE)
    torch.save(search, scale_path)
    save_json(log_path, {k: v for k, v in search.items() if k != "scales"})
print(f"search time {search['search_time_s']:.1f}s, grid={search['grid']}")

## Verify that full-precision rescaling preserves the function (before rounding)

In [ ]:
T.snap.restore(model)
ids = T.calib[:4].cuda()
with torch.no_grad():
    ref = model(input_ids=ids).logits.float()
awq.apply_awq_(model, search["scales"], awq.unit_bits_uniform(28, 4), o_mode=O_MODE, quantize=False)
with torch.no_grad():
    out = model(input_ids=ids).logits.float()
check = dict(max_abs_logit_diff=float((out - ref).abs().max()), mean_abs_logit_diff=float((out - ref).abs().mean()),
             argmax_agreement=float((out.argmax(-1) == ref.argmax(-1)).float().mean()),
             ref_logit_abs_mean=float(ref.abs().mean()))
save_json(ctx.results / "awq_rescale_equivalence.json", check)
T.snap.restore(model)
check

## Q2: AWQ-style scaling + the same Q1 quantizer

In [ ]:
for b in BITS:
    ub = awq.unit_bits_uniform(28, b)
    def apply(m, ub=ub):
        return awq.apply_awq_(m, search["scales"], ub, o_mode=O_MODE)
    T.runner.run(f"q2_awq_int{b}", apply_fn=apply,
                 storage_fn=lambda m, st, ub=ub: dict(packed=quant.model_packed_bytes(m, awq.projection_bits(ub, 28), st["explicit_scale_channels"])),
                 meta=dict(method="AWQ-style scaling + uniform group-wise PTQ", bits=b, group_size=128, o_mode=O_MODE,
                           grid=search["grid"], search_time_s=search["search_time_s"]))

## Calibration output error: Q1 (alpha = 0) vs Q2 (searched alpha)

In [ ]:
log = search["log"]
rows = []
for key, d in log.items():
    li, g = key.split(".")
    for b in (2, 3, 4, 8):
        rows.append(dict(block=int(li), group=g, bits=b, alpha=d[b]["ratio"], err_q1=d[b]["err_identity"], err_q2=d[b]["err"]))
edf = pd.DataFrame(rows)
edf["reduction"] = 1 - edf.err_q2 / edf.err_q1
edf.to_csv(ctx.results / "q1_q2_calibration_error.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, b in zip(axes, (8, 4, 3)):
    sns.heatmap(edf[edf.bits == b].pivot(index="block", columns="group", values="reduction")[["qkv", "o", "gate_up", "down"]],
                ax=ax, cmap="RdBu", center=0, vmin=-0.2, vmax=0.8)
    ax.set_title(f"INT{b}: relative calibration-error reduction by AWQ")
save_figure(ctx.figures, "q2_awq_error_reduction")
edf.groupby(["bits", "group"])[["alpha", "reduction"]].mean().unstack()

In [ ]:
from task1_compression.src.analysis import summary_table, compare_table
tab = summary_table(ctx.results)
tab.loc[["fp16"] + [f"q1_int{b}" for b in BITS] + [f"q2_awq_int{b}" for b in BITS],
        ["wikitext2_ppl", "c4_ppl", "hs_n", "hs_acc", "hs_acc_norm", "packed_MB", "dense_fp16_MB", "compression_s"]]

In [ ]:
compare_table(ctx.results, [(f"q2_awq_int{b}", f"q1_int{b}") for b in BITS] + [(f"q2_awq_int{b}", "fp16") for b in BITS])